# Sentiment Analysis Dataset - Exploratory Data Analysis

This notebook performs comprehensive exploratory data analysis on a freshly tagged sentiment analysis dataset.

**Contents:**
1. Data Loading & Initial Inspection
2. Data Quality Assessment
3. Class Distribution Analysis
4. Text Statistics & Characteristics
5. Vocabulary Analysis
6. N-gram Analysis
7. Sentiment-Specific Patterns
8. Data Visualization
9. Key Findings & Recommendations

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re
import warnings

# Text processing
from wordcloud import WordCloud
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.util import ngrams

# Statistical analysis
from scipy import stats

warnings.filterwarnings('ignore')

# Download required NLTK data
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

## 1. Data Loading & Initial Inspection

In [ ]:
# Load the dataset
# Update the file path and column names according to your dataset
FILE_PATH = 'sentiment_data.csv'  # Change this to your file path
TEXT_COLUMN = 'text'              # Column containing text data
LABEL_COLUMN = 'sentiment'        # Column containing sentiment labels

df = pd.read_csv(FILE_PATH)

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
df.head()

In [ ]:
# Basic dataset information
print("Dataset Info:")
print("=" * 50)
df.info()

print("\n" + "=" * 50)
print("\nFirst few samples:")
print("=" * 50)
for idx, row in df.head(3).iterrows():
    print(f"\nSample {idx + 1}:")
    print(f"Text: {row[TEXT_COLUMN][:100]}...")
    print(f"Sentiment: {row[LABEL_COLUMN]}")

## 2. Data Quality Assessment

In [ ]:
# Check for missing values
print("Missing Values:")
print("=" * 50)
missing_data = df.isnull().sum()
missing_percent = (missing_data / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing_data,
    'Percentage': missing_percent
})
print(missing_df[missing_df['Missing Count'] > 0])

if missing_df['Missing Count'].sum() == 0:
    print("✓ No missing values found!")

In [ ]:
# Check for duplicates
print("Duplicate Analysis:")
print("=" * 50)
duplicates_total = df.duplicated().sum()
duplicates_text = df.duplicated(subset=[TEXT_COLUMN]).sum()

print(f"Total duplicate rows: {duplicates_total} ({duplicates_total/len(df)*100:.2f}%)")
print(f"Duplicate texts: {duplicates_text} ({duplicates_text/len(df)*100:.2f}%)")

if duplicates_text > 0:
    print("\nSample duplicate texts:")
    dup_texts = df[df.duplicated(subset=[TEXT_COLUMN], keep=False)].sort_values(TEXT_COLUMN)
    print(dup_texts[[TEXT_COLUMN, LABEL_COLUMN]].head(10))

In [ ]:
# Check for empty or very short texts
df['text_length'] = df[TEXT_COLUMN].astype(str).apply(len)
df['word_count'] = df[TEXT_COLUMN].astype(str).apply(lambda x: len(x.split()))

print("Text Quality Checks:")
print("=" * 50)
print(f"Empty texts: {(df['text_length'] == 0).sum()}")
print(f"Very short texts (< 10 chars): {(df['text_length'] < 10).sum()}")
print(f"Single word texts: {(df['word_count'] == 1).sum()}")

if (df['text_length'] < 10).sum() > 0:
    print("\nSample short texts:")
    print(df[df['text_length'] < 10][[TEXT_COLUMN, LABEL_COLUMN, 'text_length']].head())

## 3. Class Distribution Analysis

In [ ]:
# Sentiment class distribution
print("Sentiment Class Distribution:")
print("=" * 50)
class_dist = df[LABEL_COLUMN].value_counts().sort_index()
class_percent = (class_dist / len(df)) * 100

dist_df = pd.DataFrame({
    'Count': class_dist,
    'Percentage': class_percent
})
print(dist_df)

# Calculate imbalance ratio
max_class = class_dist.max()
min_class = class_dist.min()
imbalance_ratio = max_class / min_class
print(f"\nImbalance Ratio: {imbalance_ratio:.2f}:1")

if imbalance_ratio > 3:
    print("⚠️  Warning: Significant class imbalance detected!")
elif imbalance_ratio > 1.5:
    print("⚡ Moderate class imbalance present")
else:
    print("✓ Classes are relatively balanced")

In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Bar plot
class_dist.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Sentiment Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Sentiment', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].tick_params(axis='x', rotation=45)

# Add value labels on bars
for i, v in enumerate(class_dist.values):
    axes[0].text(i, v + len(df)*0.01, str(v), ha='center', va='bottom', fontweight='bold')

# Pie chart
colors = sns.color_palette('husl', len(class_dist))
axes[1].pie(class_dist.values, labels=class_dist.index, autopct='%1.1f%%',
            startangle=90, colors=colors, textprops={'fontsize': 11})
axes[1].set_title('Sentiment Proportion', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 4. Text Statistics & Characteristics

In [ ]:
# Calculate additional text features
df['char_count'] = df[TEXT_COLUMN].astype(str).apply(len)
df['word_count'] = df[TEXT_COLUMN].astype(str).apply(lambda x: len(x.split()))
df['avg_word_length'] = df[TEXT_COLUMN].astype(str).apply(
    lambda x: np.mean([len(word) for word in x.split()]) if len(x.split()) > 0 else 0
)
df['sentence_count'] = df[TEXT_COLUMN].astype(str).apply(lambda x: len(re.split(r'[.!?]+', x)))
df['uppercase_count'] = df[TEXT_COLUMN].astype(str).apply(lambda x: sum(1 for c in x if c.isupper()))
df['digit_count'] = df[TEXT_COLUMN].astype(str).apply(lambda x: sum(c.isdigit() for c in x))
df['special_char_count'] = df[TEXT_COLUMN].astype(str).apply(
    lambda x: len(re.findall(r'[^a-zA-Z0-9\s]', x))
)

# Overall statistics
print("Overall Text Statistics:")
print("=" * 50)
stats_df = df[['char_count', 'word_count', 'avg_word_length', 
               'sentence_count', 'uppercase_count', 'special_char_count']].describe()
print(stats_df)

In [ ]:
# Statistics by sentiment class
print("\nText Statistics by Sentiment Class:")
print("=" * 50)
grouped_stats = df.groupby(LABEL_COLUMN)[['char_count', 'word_count', 'avg_word_length']].agg(['mean', 'median', 'std'])
print(grouped_stats.round(2))

In [ ]:
# Visualize text length distributions
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Character count distribution by sentiment
for sentiment in df[LABEL_COLUMN].unique():
    data = df[df[LABEL_COLUMN] == sentiment]['char_count']
    axes[0, 0].hist(data, bins=50, alpha=0.6, label=sentiment)
axes[0, 0].set_title('Character Count Distribution by Sentiment', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Character Count')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].legend()

# Word count distribution by sentiment
for sentiment in df[LABEL_COLUMN].unique():
    data = df[df[LABEL_COLUMN] == sentiment]['word_count']
    axes[0, 1].hist(data, bins=50, alpha=0.6, label=sentiment)
axes[0, 1].set_title('Word Count Distribution by Sentiment', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Word Count')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].legend()

# Box plot for word count by sentiment
df.boxplot(column='word_count', by=LABEL_COLUMN, ax=axes[1, 0])
axes[1, 0].set_title('Word Count Distribution by Sentiment', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Sentiment')
axes[1, 0].set_ylabel('Word Count')
plt.sca(axes[1, 0])
plt.xticks(rotation=45)

# Average word length by sentiment
df.boxplot(column='avg_word_length', by=LABEL_COLUMN, ax=axes[1, 1])
axes[1, 1].set_title('Average Word Length by Sentiment', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Sentiment')
axes[1, 1].set_ylabel('Average Word Length')
plt.sca(axes[1, 1])
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

## 5. Vocabulary Analysis

In [ ]:
# Tokenization and vocabulary building
def tokenize_text(text):
    """Tokenize and clean text"""
    text = str(text).lower()
    tokens = word_tokenize(text)
    # Remove non-alphabetic tokens and single characters
    tokens = [token for token in tokens if token.isalpha() and len(token) > 1]
    return tokens

df['tokens'] = df[TEXT_COLUMN].apply(tokenize_text)

# Overall vocabulary
all_tokens = [token for tokens in df['tokens'] for token in tokens]
vocab = set(all_tokens)

print("Vocabulary Statistics:")
print("=" * 50)
print(f"Total tokens: {len(all_tokens):,}")
print(f"Unique tokens (vocabulary size): {len(vocab):,}")
print(f"Vocabulary richness (unique/total): {len(vocab)/len(all_tokens):.4f}")

In [ ]:
# Most common words overall
word_freq = Counter(all_tokens)
stop_words = set(stopwords.words('english'))

# Filter out stopwords
filtered_freq = {word: freq for word, freq in word_freq.items() if word not in stop_words}

print("Top 20 Most Common Words (excluding stopwords):")
print("=" * 50)
for word, freq in Counter(filtered_freq).most_common(20):
    print(f"{word:20s}: {freq:6d}")

In [ ]:
# Vocabulary by sentiment
print("\nVocabulary by Sentiment:")
print("=" * 50)

sentiment_vocab = {}
sentiment_word_freq = {}

for sentiment in df[LABEL_COLUMN].unique():
    sentiment_tokens = [token for tokens in df[df[LABEL_COLUMN] == sentiment]['tokens'] for token in tokens]
    sentiment_vocab[sentiment] = set(sentiment_tokens)
    sentiment_word_freq[sentiment] = Counter(sentiment_tokens)
    
    print(f"\n{sentiment}:")
    print(f"  Total tokens: {len(sentiment_tokens):,}")
    print(f"  Unique tokens: {len(sentiment_vocab[sentiment]):,}")
    print(f"  Vocabulary richness: {len(sentiment_vocab[sentiment])/len(sentiment_tokens):.4f}")

In [ ]:
# Top words per sentiment (excluding stopwords)
print("\nTop 15 Words by Sentiment (excluding stopwords):")
print("=" * 70)

for sentiment in df[LABEL_COLUMN].unique():
    print(f"\n{sentiment.upper()}:")
    filtered_sentiment_freq = {word: freq for word, freq in sentiment_word_freq[sentiment].items() 
                               if word not in stop_words}
    top_words = Counter(filtered_sentiment_freq).most_common(15)
    for word, freq in top_words:
        print(f"  {word:20s}: {freq:6d}")

In [ ]:
# Visualize top words by sentiment
sentiments = df[LABEL_COLUMN].unique()
n_sentiments = len(sentiments)
n_cols = min(3, n_sentiments)
n_rows = (n_sentiments + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(7*n_cols, 5*n_rows))
if n_sentiments == 1:
    axes = [axes]
elif n_rows == 1:
    axes = axes
else:
    axes = axes.flatten()

for idx, sentiment in enumerate(sentiments):
    filtered_sentiment_freq = {word: freq for word, freq in sentiment_word_freq[sentiment].items() 
                               if word not in stop_words}
    top_words = Counter(filtered_sentiment_freq).most_common(15)
    words, freqs = zip(*top_words) if top_words else ([], [])
    
    axes[idx].barh(range(len(words)), freqs, color='steelblue')
    axes[idx].set_yticks(range(len(words)))
    axes[idx].set_yticklabels(words)
    axes[idx].invert_yaxis()
    axes[idx].set_xlabel('Frequency')
    axes[idx].set_title(f'Top Words - {sentiment}', fontsize=12, fontweight='bold')

# Hide extra subplots
for idx in range(n_sentiments, len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Word clouds by sentiment
sentiments = df[LABEL_COLUMN].unique()
n_sentiments = len(sentiments)
n_cols = min(3, n_sentiments)
n_rows = (n_sentiments + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(7*n_cols, 5*n_rows))
if n_sentiments == 1:
    axes = [axes]
elif n_rows == 1:
    axes = axes
else:
    axes = axes.flatten()

for idx, sentiment in enumerate(sentiments):
    sentiment_text = ' '.join(df[df[LABEL_COLUMN] == sentiment][TEXT_COLUMN].astype(str))
    
    wordcloud = WordCloud(width=800, height=400, 
                          background_color='white',
                          stopwords=stop_words,
                          max_words=100,
                          colormap='viridis').generate(sentiment_text)
    
    axes[idx].imshow(wordcloud, interpolation='bilinear')
    axes[idx].axis('off')
    axes[idx].set_title(f'Word Cloud - {sentiment}', fontsize=14, fontweight='bold', pad=20)

# Hide extra subplots
for idx in range(n_sentiments, len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

## 6. N-gram Analysis

In [ ]:
# Extract bigrams and trigrams
def get_ngrams(tokens_list, n=2):
    """Extract n-grams from token list"""
    all_ngrams = []
    for tokens in tokens_list:
        # Filter out stopwords
        filtered_tokens = [t for t in tokens if t not in stop_words]
        if len(filtered_tokens) >= n:
            all_ngrams.extend(list(ngrams(filtered_tokens, n)))
    return all_ngrams

# Overall bigrams
all_bigrams = get_ngrams(df['tokens'], n=2)
bigram_freq = Counter(all_bigrams)

print("Top 20 Bigrams (excluding stopwords):")
print("=" * 50)
for bigram, freq in bigram_freq.most_common(20):
    print(f"{' '.join(bigram):30s}: {freq:6d}")

In [ ]:
# Trigrams
all_trigrams = get_ngrams(df['tokens'], n=3)
trigram_freq = Counter(all_trigrams)

print("\nTop 20 Trigrams (excluding stopwords):")
print("=" * 50)
for trigram, freq in trigram_freq.most_common(20):
    print(f"{' '.join(trigram):40s}: {freq:6d}")

In [ ]:
# Bigrams by sentiment
print("\nTop 10 Bigrams by Sentiment:")
print("=" * 70)

sentiment_bigrams = {}
for sentiment in df[LABEL_COLUMN].unique():
    sentiment_tokens = df[df[LABEL_COLUMN] == sentiment]['tokens']
    sentiment_bigrams[sentiment] = get_ngrams(sentiment_tokens, n=2)
    
    print(f"\n{sentiment.upper()}:")
    bigram_freq = Counter(sentiment_bigrams[sentiment])
    for bigram, freq in bigram_freq.most_common(10):
        print(f"  {' '.join(bigram):30s}: {freq:6d}")

In [ ]:
# Visualize top bigrams by sentiment
sentiments = df[LABEL_COLUMN].unique()
n_sentiments = len(sentiments)
n_cols = min(3, n_sentiments)
n_rows = (n_sentiments + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(7*n_cols, 5*n_rows))
if n_sentiments == 1:
    axes = [axes]
elif n_rows == 1:
    axes = axes
else:
    axes = axes.flatten()

for idx, sentiment in enumerate(sentiments):
    bigram_freq = Counter(sentiment_bigrams[sentiment])
    top_bigrams = bigram_freq.most_common(15)
    
    if top_bigrams:
        bigrams, freqs = zip(*top_bigrams)
        bigram_labels = [' '.join(bg) for bg in bigrams]
        
        axes[idx].barh(range(len(bigrams)), freqs, color='coral')
        axes[idx].set_yticks(range(len(bigrams)))
        axes[idx].set_yticklabels(bigram_labels, fontsize=9)
        axes[idx].invert_yaxis()
        axes[idx].set_xlabel('Frequency')
        axes[idx].set_title(f'Top Bigrams - {sentiment}', fontsize=12, fontweight='bold')

# Hide extra subplots
for idx in range(n_sentiments, len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

## 7. Sentiment-Specific Patterns

In [ ]:
# Check for sentiment-indicative patterns
patterns = {
    'exclamation': r'!',
    'question': r'\?',
    'all_caps_words': r'\b[A-Z]{2,}\b',
    'emoticons_positive': r'[;:]-?[)D]|[)D]',
    'emoticons_negative': r'[;:]-?[(<]|[(<]',
    'ellipsis': r'\.{2,}',
}

for pattern_name, pattern in patterns.items():
    df[pattern_name] = df[TEXT_COLUMN].astype(str).apply(
        lambda x: len(re.findall(pattern, x))
    )

print("Pattern Analysis by Sentiment:")
print("=" * 70)
pattern_stats = df.groupby(LABEL_COLUMN)[list(patterns.keys())].mean()
print(pattern_stats.round(3))

In [ ]:
# Visualize pattern usage by sentiment
pattern_stats_transpose = pattern_stats.T

fig, ax = plt.subplots(figsize=(12, 6))
pattern_stats_transpose.plot(kind='bar', ax=ax, width=0.8)
ax.set_title('Average Pattern Usage by Sentiment', fontsize=14, fontweight='bold')
ax.set_xlabel('Pattern Type', fontsize=12)
ax.set_ylabel('Average Count per Text', fontsize=12)
ax.legend(title='Sentiment', title_fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Statistical tests for significant differences
print("\nStatistical Significance Tests (ANOVA):")
print("=" * 70)
print("Testing if text characteristics differ significantly across sentiments\n")

features_to_test = ['word_count', 'char_count', 'avg_word_length', 'exclamation', 'question']

for feature in features_to_test:
    groups = [df[df[LABEL_COLUMN] == sentiment][feature].values 
              for sentiment in df[LABEL_COLUMN].unique()]
    f_stat, p_value = stats.f_oneway(*groups)
    
    significance = "***" if p_value < 0.001 else "**" if p_value < 0.01 else "*" if p_value < 0.05 else "ns"
    print(f"{feature:25s}: F={f_stat:8.2f}, p={p_value:.4f} {significance}")

print("\n*** p<0.001, ** p<0.01, * p<0.05, ns=not significant")

## 8. Data Visualization Summary

In [ ]:
# Correlation heatmap of text features
feature_cols = ['word_count', 'char_count', 'avg_word_length', 'sentence_count', 
                'uppercase_count', 'special_char_count', 'exclamation', 'question']

plt.figure(figsize=(12, 10))
correlation_matrix = df[feature_cols].corr()
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=1)
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# Text length vs sentiment scatter plot with density
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter plot
for sentiment in df[LABEL_COLUMN].unique():
    subset = df[df[LABEL_COLUMN] == sentiment]
    axes[0].scatter(subset['word_count'], subset['char_count'], 
                   alpha=0.5, label=sentiment, s=20)
axes[0].set_xlabel('Word Count', fontsize=12)
axes[0].set_ylabel('Character Count', fontsize=12)
axes[0].set_title('Text Length Distribution by Sentiment', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Violin plot
sentiment_order = sorted(df[LABEL_COLUMN].unique())
sns.violinplot(data=df, x=LABEL_COLUMN, y='word_count', ax=axes[1], order=sentiment_order)
axes[1].set_xlabel('Sentiment', fontsize=12)
axes[1].set_ylabel('Word Count', fontsize=12)
axes[1].set_title('Word Count Distribution by Sentiment', fontsize=14, fontweight='bold')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

## 9. Key Findings & Recommendations

In [ ]:
# Generate summary report
print("=" * 70)
print("EXPLORATORY DATA ANALYSIS SUMMARY")
print("=" * 70)

print("\n1. DATASET OVERVIEW")
print("-" * 70)
print(f"Total samples: {len(df):,}")
print(f"Number of sentiment classes: {df[LABEL_COLUMN].nunique()}")
print(f"Sentiment classes: {', '.join(map(str, sorted(df[LABEL_COLUMN].unique())))}")

print("\n2. DATA QUALITY")
print("-" * 70)
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Duplicate texts: {duplicates_text} ({duplicates_text/len(df)*100:.2f}%)")
print(f"Very short texts (<10 chars): {(df['text_length'] < 10).sum()}")

print("\n3. CLASS BALANCE")
print("-" * 70)
for sentiment, count in class_dist.items():
    print(f"{sentiment}: {count:,} ({count/len(df)*100:.1f}%)")
print(f"Imbalance ratio: {imbalance_ratio:.2f}:1")

print("\n4. TEXT CHARACTERISTICS")
print("-" * 70)
print(f"Average word count: {df['word_count'].mean():.1f} ± {df['word_count'].std():.1f}")
print(f"Average character count: {df['char_count'].mean():.1f} ± {df['char_count'].std():.1f}")
print(f"Average word length: {df['avg_word_length'].mean():.2f} letters")

print("\n5. VOCABULARY")
print("-" * 70)
print(f"Total unique words: {len(vocab):,}")
print(f"Vocabulary richness: {len(vocab)/len(all_tokens):.4f}")
print(f"Most common word: '{Counter(filtered_freq).most_common(1)[0][0]}' ({Counter(filtered_freq).most_common(1)[0][1]} occurrences)")

print("\n6. KEY RECOMMENDATIONS")
print("-" * 70)

recommendations = []

# Class imbalance
if imbalance_ratio > 3:
    recommendations.append(
        "⚠️  CRITICAL: Significant class imbalance detected. Consider:\n"
        "   - Using stratified sampling for train/test split\n"
        "   - Applying SMOTE or other oversampling techniques\n"
        "   - Using class weights in model training\n"
        "   - Evaluating with F1-score and confusion matrix, not just accuracy"
    )

# Duplicates
if duplicates_text > len(df) * 0.05:
    recommendations.append(
        f"⚠️  {duplicates_text} duplicate texts found ({duplicates_text/len(df)*100:.1f}%).\n"
        "   Consider removing duplicates to prevent data leakage."
    )

# Text length variation
if df['word_count'].std() / df['word_count'].mean() > 1:
    recommendations.append(
        "⚡ High variance in text length detected.\n"
        "   Consider truncation/padding strategies for neural network models."
    )

# Vocabulary size
if len(vocab) > 50000:
    recommendations.append(
        f"⚡ Large vocabulary ({len(vocab):,} unique words).\n"
        "   Consider implementing vocabulary size limits or using subword tokenization."
    )

# General recommendations
recommendations.append(
    "✓ Consider the following preprocessing steps:\n"
    "   - Text normalization (lowercasing, removing special characters)\n"
    "   - Stopword removal (if appropriate for your task)\n"
    "   - Lemmatization or stemming\n"
    "   - Handling of URLs, mentions, hashtags if present"
)

recommendations.append(
    "✓ Model selection suggestions:\n"
    "   - Start with simple baselines (Naive Bayes, Logistic Regression)\n"
    "   - Consider ensemble methods (Random Forest, XGBoost)\n"
    "   - Explore deep learning (LSTM, BERT) if computational resources allow\n"
    "   - Use cross-validation for robust evaluation"
)

for i, rec in enumerate(recommendations, 1):
    print(f"\n{i}. {rec}")

print("\n" + "=" * 70)
print("END OF ANALYSIS")
print("=" * 70)

In [ ]:
# Export summary statistics to CSV
summary_stats = df.groupby(LABEL_COLUMN).agg({
    'word_count': ['mean', 'median', 'std', 'min', 'max'],
    'char_count': ['mean', 'median', 'std', 'min', 'max'],
    'avg_word_length': ['mean', 'median', 'std'],
    TEXT_COLUMN: 'count'
}).round(2)

summary_stats.to_csv('sentiment_eda_summary.csv')
print("✓ Summary statistics exported to 'sentiment_eda_summary.csv'")